# 🧪 Kairos CIDA — Release Readiness Benchmark Suite

Этот ноутбук — отдельный **evaluation-раннер**, не тренировочный.
Задача: загрузить все поколения CIDA (v1 Prototype, v2 Rollback, v4 SFT, v5 RL), подключить их к соответствующим базовым моделям и прогнать **широкую батарею бенчмарков** по разным концептам — от логики и математики до галлюцинаций и агентности — чтобы получить объективную и честную картину готовности к релизу.

**Сравниваются следующие варианты на одних и тех же вопросах:**
1. `Base Qwen2.5-1.5B`, `Base Qwen1.5-1.8B`, `Base Qwen1.5-7B` — базовые модели без CIDA (baselines)
2. `CIDA v1 (Prototype, 7B)` — первая версия на Qwen1.5-7B
3. `CIDA v2 (Rollback, 1.8B)` — вторая версия на Qwen1.5-1.8B
4. `CIDA v4 (SFT, 1.5B)` — четвертая версия (SFT) на Qwen2.5-1.5B (если найдены веса)
5. `CIDA v5 (RL, 1.5B)` — пятая версия (RL) на Qwen2.5-1.5B (если найдены веса)

**Категории бенчмарков:**

| Категория | Что проверяет | Датасеты |
|---|---|---|
| Formal Logic | дедукция, валидность аргументов | BBH: formal_fallacies, logical_deduction, navigate |
| Natural Language Inference | вывод из посылок | FOLIO |
| Math Reasoning | пошаговая математика | GSM8K, MATH (competition_math) |
| World Knowledge | факты, эрудиция | MMLU (подвыборка по 5 доменам) |
| Commonsense | бытовая логика | HellaSwag, ARC-Challenge, CommonsenseQA |
| Truthfulness / Hallucination | склонность придумывать факты | TruthfulQA (MC1) |
| Agentic Planning | принятие решений при конфликте ограничений | синтетический constraint-satisfaction набор |
| Tool-Use / Function-Calling | выбор правильного инструмента/аргументов | Salesforce xLAM function-calling (подвыборка) |

Плюс отдельно меряется **задержка инференса** (base vs CIDA) и считается **итоговый scorecard** относительно критериев готовности к релизу.

> ⚠️ Для каждого датасета используется `try/except` — если что-то не загрузится (сеть, переименование датасета на HF), тот блок просто пропускается и это видно в логе, весь остальной прогон не ломается.


In [ ]:
# === 0. Установка зависимостей ===
!pip install -q torch transformers datasets accelerate bitsandbytes huggingface_hub gdown matplotlib numpy tqdm

import getpass, sys, os, subprocess

# Клонируем приватный плагин (тот же репозиторий, что и в тренировочном ноутбуке)
if not os.path.exists("CIDA-plugin"):
    token = getpass.getpass("GitHub PAT (Enter — если репозиторий публичный/уже доступен): ").strip()
    if token:
        os.system(f"git clone https://{token}@github.com/Kairatzh/CIDA-plugin.git")
    else:
        os.system("git clone https://github.com/Kairatzh/CIDA-plugin.git")
    sys.path.append("CIDA-plugin")
    os.system("pip install -q -e CIDA-plugin")
else:
    sys.path.append("CIDA-plugin")

import torch
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")


In [ ]:
# === 1. Google Drive: подключение и поиск весов ===
# Работает и в Colab (mount), и локально (просто укажи GDRIVE_ZIP_URL / локальные папки).

import os

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive"
else:
    DRIVE_ROOT = os.environ.get("DRIVE_ROOT", "./drive_mirror")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f"Не в Colab — ожидаю веса локально в: {DRIVE_ROOT}")

# Если у тебя веса лежат как ОДИН файл/шаренная ссылка на Google Drive (а не как папка
# внутри твоего Drive), впиши сюда file id или полный share-ссылку и раскомментируй:
GDRIVE_ZIP_URL = None  # например: "https://drive.google.com/uc?id=<FILE_ID>"

if GDRIVE_ZIP_URL:
    import gdown
    os.makedirs("downloaded_weights", exist_ok=True)
    zip_path = "downloaded_weights/cida_weights.zip"
    gdown.download(GDRIVE_ZIP_URL, zip_path, quiet=False)
    os.system(f"unzip -oq {zip_path} -d downloaded_weights")

# Кандидаты на путь к весам — порядок = приоритет (RL > SFT), Drive > локально > скачанное
CANDIDATE_DIRS = {
    "CIDA v5 (RL)": [
        os.path.join(DRIVE_ROOT, "cida_v5_rl_weights", "best"),
        os.path.join(DRIVE_ROOT, "cida_v5_rl_weights"),
        "cida_v5_rl_weights/best",
        "cida_v5_rl_weights",
        "downloaded_weights/cida_v5_rl_weights",
    ],
    "CIDA v4 (SFT)": [
        os.path.join(DRIVE_ROOT, "cida_v4_universal_weights"),
        "cida_v4_universal_weights",
        "downloaded_weights/cida_v4_universal_weights",
    ],
}

def find_weights(paths):
    for p in paths:
        if os.path.exists(os.path.join(p, "pytorch_model.bin")):
            return p
    return None

weights_found = {}
for name, paths in CANDIDATE_DIRS.items():
    found = find_weights(paths)
    if found:
        weights_found[name] = found
        print(f"✅ Найдены веса {name}: {found}")
    else:
        print(f"⚠️ Веса {name} не найдены (проверены: {paths})")

if not weights_found:
    print("\n❌ Ни один набор весов CIDA не найден. Проверь, что Drive смонтирован")
    print("   и что путь/название папки совпадает с тем, что сохранял тренировочный ноутбук.")


In [ ]:
# === 2. Загрузка моделей + CIDA ===
import torch
import gc
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Конфигурация всех вариантов CIDA и их базовых моделей
VARIANTS_CONFIG = {
    "Base Qwen2.5-1.5B": {
        "backbone": "Qwen/Qwen2.5-1.5B-Instruct",
        "cida_source": None,
        "is_v4": False,
        "is_hf": False,
        "load_base_only": True
    },
    "Base Qwen1.5-1.8B": {
        "backbone": "Qwen/Qwen1.5-1.8B-Chat",
        "cida_source": None,
        "is_v4": False,
        "is_hf": False,
        "load_base_only": True
    },
    "Base Qwen1.5-7B": {
        "backbone": "Qwen/Qwen1.5-7B-Chat",
        "cida_source": None,
        "is_v4": False,
        "is_hf": False,
        "load_base_only": True
    },
    "CIDA v1 (Prototype, 7B)": {
        "backbone": "Qwen/Qwen1.5-7B-Chat",
        "cida_source": "kirtzh/Kairos-7B-CIDA-v1.0",
        "is_v4": False,
        "is_hf": True,
        "load_base_only": False,
        "base_variant": "Base Qwen1.5-7B"
    },
    "CIDA v2 (Rollback, 1.8B)": {
        "backbone": "Qwen/Qwen1.5-1.8B-Chat",
        "cida_source": "kirtzh/Kairos-CIDA-1.3B-v1.0",
        "is_v4": False,
        "is_hf": True,
        "load_base_only": False,
        "base_variant": "Base Qwen1.5-1.8B"
    }
}

# Добавляем v4 и v5, если их локальные пути найдены на диске/Drive
if "CIDA v4 (SFT)" in weights_found:
    VARIANTS_CONFIG["CIDA v4 (SFT, 1.5B)"] = {
        "backbone": "Qwen/Qwen2.5-1.5B-Instruct",
        "cida_source": weights_found["CIDA v4 (SFT)"],
        "is_v4": True,
        "is_hf": False,
        "load_base_only": False,
        "base_variant": "Base Qwen2.5-1.5B"
    }
else:
    print("⚠️ CIDA v4 (SFT) weights not found. Skipping from evaluation config.")

if "CIDA v5 (RL)" in weights_found:
    VARIANTS_CONFIG["CIDA v5 (RL, 1.5B)"] = {
        "backbone": "Qwen/Qwen2.5-1.5B-Instruct",
        "cida_source": weights_found["CIDA v5 (RL)"],
        "is_v4": True,
        "is_hf": False,
        "load_base_only": False,
        "base_variant": "Base Qwen2.5-1.5B"
    }
else:
    print("⚠️ CIDA v5 (RL) weights not found. Skipping from evaluation config.")

print("\nConfigured variants for evaluation:")
for name, vcfg in VARIANTS_CONFIG.items():
    print(f"  • {name} (Backbone: {vcfg['backbone']}, CIDA: {vcfg['cida_source']})")


In [ ]:
# === 3. Универсальные функции генерации + замер задержки ===
import time
import re
import torch.nn as nn
from huggingface_hub import hf_hub_download

# Проектор для старых версий (v1/v2)
class CIDAPlanProjectorOld(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan_tokens=8):
        super().__init__()
        self.num_plan_tokens = num_plan_tokens
        self.d_llm = d_llm
        d_flat = num_agents * d_cida
        self.projector = nn.Sequential(
            nn.Linear(d_flat, d_flat*2), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(d_flat*2, num_plan_tokens*d_llm),
            nn.LayerNorm(num_plan_tokens*d_llm))
    def forward(self, state_s):
        B = state_s.size(0)
        return self.projector(state_s.reshape(B,-1).float()).view(B, self.num_plan_tokens, self.d_llm)

# Проектор для новых версий (v4/v5)
class LatentAEProjector(nn.Module):
    def __init__(self, d_cida, d_llm, num_agents, num_plan=8, num_heads=8):
        super().__init__()
        self.num_plan = num_plan
        self.plan_queries = nn.Parameter(torch.randn(1, num_plan, d_cida) * 0.02)
        self.cross_attn = nn.MultiheadAttention(d_cida, num_heads, batch_first=True, dropout=0.1)
        self.cross_norm = nn.LayerNorm(d_cida)
        self.self_attn = nn.MultiheadAttention(d_cida, num_heads, batch_first=True, dropout=0.1)
        self.self_norm = nn.LayerNorm(d_cida)
        self.ffn = nn.Sequential(
            nn.Linear(d_cida, d_cida * 2), nn.SiLU(),
            nn.Linear(d_cida * 2, d_cida)
        )
        self.ffn_norm = nn.LayerNorm(d_cida)
        self.out_proj = nn.Sequential(
            nn.Linear(d_cida, d_llm * 2), nn.SiLU(),
            nn.Linear(d_llm * 2, d_llm), nn.LayerNorm(d_llm)
        )
        self.gate = nn.Parameter(torch.tensor(0.1))
    def forward(self, agent_states):
        B = agent_states.size(0)
        q = self.plan_queries.expand(B, -1, -1)
        ca, _ = self.cross_attn(q, agent_states, agent_states)
        q = self.cross_norm(q + ca)
        sa, _ = self.self_attn(q, q, q)
        q = self.self_norm(q + sa)
        q = self.ffn_norm(q + self.ffn(q))
        return self.out_proj(q) * self.gate.sigmoid()

SYSTEM_PROMPT = "Solve logically."

def build_prompt(q):
    return f"<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n<|im_start|>user\n{q}\n<|im_end|>\n<|im_start|>assistant\n"

def clean_cida_output(text, prompt_question=None):
    text = re.sub(r'^zlib\\_compression\s*', '', text, flags=re.IGNORECASE).strip()
    changed = True
    while changed:
        changed = False
        text = text.strip()
        if text.lower().startswith(("\\n", "\\r", "\\t")):
            text = text[2:].strip()
            changed = True
            continue
        m = re.match(r'^[zn][0-9]?[\\s\\n\\r\\t\\#\\-\\:\>\\.\\,\\=\\"\\\'\\(\\)\\[\\]\/\\\\]+', text, re.IGNORECASE)
        if m:
            text = text[m.end():].strip()
            changed = True
            continue
        m = re.match(r'^[zn]([A-Z0-9])', text)
        if m:
            text = text[1:].strip()
            changed = True
            continue
        m = re.match(r'^[^a-zA-Z0-9\s]+', text)
        if m:
            text = text[m.end():].strip()
            changed = True
            continue
    if prompt_question:
        pq = prompt_question.strip().lower()
        pq_clean = re.sub(r'[^a-z0-9\s]', '', pq)
        text_lower = text.lower()
        if pq_clean in re.sub(r'[^a-z0-9\s]', '', text_lower)[:len(pq_clean) + 20]:
            for length in range(len(prompt_question), 10, -1):
                part = prompt_question[:length]
                if part in text:
                    idx = text.index(part) + len(part)
                    text = text[idx:].strip()
                    text = re.sub(r'^[:\s\-\>]+', '', text).strip()
                    break
    text = re.sub(r'^[^a-zA-Z0-9\s]+', '', text).strip()
    return text

@torch.no_grad()
def generate_single(q, llm, tokenizer, cida=None, proj=None, is_cida=False, is_v4=False, num_plan=8, max_new_tokens=150, alpha=2.2, use_steering=True):
    enc = tokenizer(build_prompt(q), return_tensors="pt").to(DEVICE)
    t0 = time.time()
    
    if not is_cida:
        ids = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        dt = time.time() - t0
        text = tokenizer.decode(ids[0, enc.input_ids.size(1):], skip_special_tokens=True).strip()
        return text, dt
    
    out = llm(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
    compute_dtype = out.hidden_states[-1].dtype
    h_base = out.hidden_states[-1][0, -1].float().unsqueeze(0)

    co = cida(pooled=h_base)
    s_final = co["state_final"].s if hasattr(co["state_final"], 's') else co["state_final"]
    plan = proj(s_final)
    
    disagreement = co.get("disagreement", torch.tensor(0.0)).mean().item()
    plan = plan * alpha

    p_emb = llm.get_input_embeddings()(enc.input_ids).detach()
    cat = torch.cat([p_emb.to(compute_dtype), plan.to(compute_dtype)], dim=1)
    pm = torch.ones(1, plan.size(1), dtype=enc.attention_mask.dtype, device=DEVICE)
    gm = torch.cat([enc.attention_mask, pm], dim=1)

    if use_steering and disagreement > 0.05:
        temp = min(0.2 + disagreement * 1.5, 0.8)
        ids = llm.generate(
            inputs_embeds=cat, 
            attention_mask=gm, 
            max_new_tokens=max_new_tokens, 
            do_sample=True, 
            temperature=temp,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id
        )
    else:
        ids = llm.generate(
            inputs_embeds=cat, 
            attention_mask=gm, 
            max_new_tokens=max_new_tokens, 
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    dt = time.time() - t0
    text = tokenizer.decode(ids[0], skip_special_tokens=True).strip()
    text = clean_cida_output(text, prompt_question=q)
    return text, dt

print("✅ Generation helpers ready with Plan Amplification and Iterative Output Cleaning.")


## 📚 Загрузка бенчмарков по категориям

Каждый датасет грузится независимо (`try/except`), берём небольшую фиксированную
подвыборку (`EVAL_SAMPLES`) на каждый — это runtime-friendly проверка готовности,
а не полный академический eval. Увеличь `EVAL_SAMPLES`, если нужна более точная
оценка (за счёт времени).

In [ ]:
# === 4. Загрузка датасетов по категориям ===
from datasets import load_dataset
import random

random.seed(42)
EVAL_SAMPLES = 15  # на каждый датасет; увеличь для более надёжной статистики

benchmarks = {}  # category -> {dataset_label: [{"q":..., "a":...}, ...]}

def add_bench(category, label, items):
    benchmarks.setdefault(category, {})
    benchmarks[category][label] = items
    print(f"  ✅ {category} / {label}: {len(items)} samples")

def sample_n(ds, n):
    idxs = list(range(len(ds)))
    random.shuffle(idxs)
    return [ds[i] for i in idxs[:n]]

# --- Formal Logic (BBH) ---
print("Loading Formal Logic (BBH)...")
try:
    ds = load_dataset("lukaemon/bbh", "formal_fallacies", split="test")
    s = sample_n(ds, EVAL_SAMPLES)
    add_bench("Formal Logic", "BBH Fallacies",
        [{"q": f"Deductive Argument:\n{x['input']}\nIs this argument valid or invalid? Answer with just the word.",
          "a": x["target"].strip()} for x in s])
except Exception as e:
    print(f"  ⚠️ BBH Fallacies failed: {e}")

try:
    ds = load_dataset("lukaemon/bbh", "logical_deduction_five_objects", split="test")
    s = sample_n(ds, EVAL_SAMPLES)
    add_bench("Formal Logic", "BBH Logical Deduction",
        [{"q": f"Logic Puzzle:\n{x['input']}\nJust answer with the correct letter.",
          "a": x["target"].strip("()")} for x in s])
except Exception as e:
    print(f"  ⚠️ BBH Logical Deduction failed: {e}")

try:
    ds = load_dataset("lukaemon/bbh", "navigate", split="test")
    s = sample_n(ds, EVAL_SAMPLES)
    add_bench("Formal Logic", "BBH Navigate",
        [{"q": f"{x['input']}\nAnswer with just Yes or No.", "a": x["target"].strip()} for x in s])
except Exception as e:
    print(f"  ⚠️ BBH Navigate failed: {e}")

# --- Natural Language Inference ---
print("Loading NLI (FOLIO)...")
try:
    ds = load_dataset("tasksource/folio", split="validation")
    s = sample_n(ds, EVAL_SAMPLES)
    items = []
    for x in s:
        prems = x["premises"]
        prems_str = "\n".join(prems) if isinstance(prems, list) else str(prems)
        items.append({"q": f"Premises:\n{prems_str}\nConclusion: {x['conclusion']}\nIs the conclusion True, False, or Unknown? Answer with just the word.",
                       "a": x["label"].strip()})
    add_bench("NLI", "FOLIO", items)
except Exception as e:
    print(f"  ⚠️ FOLIO failed: {e}")

# --- Math Reasoning ---
print("Loading Math (GSM8K, MATH)...")
try:
    ds = load_dataset("openai/gsm8k", "main", split="test")
    s = sample_n(ds, EVAL_SAMPLES)
    items = []
    for x in s:
        final = x["answer"].split("####")[-1].strip()
        items.append({"q": f"{x['question']}\nGive only the final numeric answer.", "a": final})
    add_bench("Math", "GSM8K", items)
except Exception as e:
    print(f"  ⚠️ GSM8K failed: {e}")

try:
    ds = load_dataset("hendrycks/competition_math", split="test", trust_remote_code=True)
    s = sample_n(ds, EVAL_SAMPLES)
    items = []
    for x in s:
        m = re.search(r"\\boxed\{([^}]*)\}", x["solution"]) if "solution" in x else None
        final = m.group(1) if m else x.get("solution", "")[-20:]
        items.append({"q": f"{x['problem']}\nGive only the final answer.", "a": final})
    add_bench("Math", "MATH (competition)", items)
except Exception as e:
    print(f"  ⚠️ MATH failed: {e}")

# --- World Knowledge (MMLU, 5 domains) ---
print("Loading Knowledge (MMLU subset)...")
MMLU_DOMAINS = ["high_school_physics", "high_school_biology", "world_religions", "marketing", "astronomy"]
for dom in MMLU_DOMAINS:
    try:
        ds = load_dataset("cais/mmlu", dom, split="test")
        s = sample_n(ds, min(EVAL_SAMPLES, len(ds)))
        letters = "ABCD"
        items = []
        for x in s:
            choices_str = "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(x["choices"]))
            items.append({"q": f"{x['question']}\n{choices_str}\nAnswer with just the letter.",
                           "a": letters[x["answer"]]})
        add_bench("World Knowledge", f"MMLU:{dom}", items)
    except Exception as e:
        print(f"  ⚠️ MMLU {dom} failed: {e}")

# --- Commonsense ---
print("Loading Commonsense (ARC, CommonsenseQA)...")
try:
    ds = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
    s = sample_n(ds, EVAL_SAMPLES)
    items = []
    for x in s:
        labels = x["choices"]["label"]
        texts = x["choices"]["text"]
        choices_str = "\n".join(f"{l}. {t}" for l, t in zip(labels, texts))
        items.append({"q": f"{x['question']}\n{choices_str}\nAnswer with just the letter.", "a": x["answerKey"]})
    add_bench("Commonsense", "ARC-Challenge", items)
except Exception as e:
    print(f"  ⚠️ ARC-Challenge failed: {e}")

try:
    ds = load_dataset("tau/commonsense_qa", split="validation")
    s = sample_n(ds, EVAL_SAMPLES)
    items = []
    for x in s:
        labels = x["choices"]["label"]
        texts = x["choices"]["text"]
        choices_str = "\n".join(f"{l}. {t}" for l, t in zip(labels, texts))
        items.append({"q": f"{x['question']}\n{choices_str}\nAnswer with just the letter.", "a": x["answerKey"]})
    add_bench("Commonsense", "CommonsenseQA", items)
except Exception as e:
    print(f"  ⚠️ CommonsenseQA failed: {e}")

# --- Truthfulness / Hallucination ---
print("Loading Truthfulness (TruthfulQA)...")
try:
    ds = load_dataset("truthful_qa", "multiple_choice", split="validation")
    s = sample_n(ds, EVAL_SAMPLES)
    items = []
    for x in s:
        choices = x["mc1_targets"]["choices"]
        labels_arr = x["mc1_targets"]["labels"]
        correct_idx = labels_arr.index(1)
        letters = "ABCDEFGH"[:len(choices)]
        choices_str = "\n".join(f"{letters[i]}. {c}" for i, c in enumerate(choices))
        items.append({"q": f"{x['question']}\n{choices_str}\nAnswer with just the letter of the single correct/true statement.",
                       "a": letters[correct_idx]})
    add_bench("Truthfulness / Hallucination", "TruthfulQA (MC1)", items)
except Exception as e:
    print(f"  ⚠️ TruthfulQA failed: {e}")

# --- Agentic Planning (synthetic constraint-satisfaction) ---
print("Building synthetic Agentic Planning set...")
random.seed(7)
CITIES = ["Astana", "Almaty", "Dubai", "Istanbul", "Tashkent", "Baku", "Tbilisi"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
def make_planning_item():
    origin, dest = random.sample(CITIES, 2)
    budget = random.choice([700, 900, 1100, 1400])
    req_day = random.choice(DAYS)
    n_opts = 3
    letters = "ABC"
    opts = []
    correct = None
    for i in range(n_opts):
        day = random.choice(DAYS)
        price = random.choice([600, 800, 950, 1200, 1500])
        valid = (day == req_day and price < budget)
        if valid and correct is None:
            correct = letters[i]
        opts.append((letters[i], day, price))
    if correct is None:
        # force one valid option
        idx = random.randrange(n_opts)
        opts[idx] = (letters[idx], req_day, budget - 100)
        correct = letters[idx]
    opts_str = "\n".join(f"{l}: depart {d}, price ${p}" for l, d, p in opts)
    q = (f"Book a flight from {origin} to {dest}. Constraints: budget strictly under ${budget}, "
         f"must depart on {req_day}.\nOptions:\n{opts_str}\nWhich option satisfies BOTH constraints? Answer with just the letter.")
    return {"q": q, "a": correct}

add_bench("Agentic Planning", "Synthetic Constraint-Satisfaction",
          [make_planning_item() for _ in range(EVAL_SAMPLES)])

# --- Tool-Use / Function-Calling ---
print("Loading Tool-Use (xLAM function-calling)...")
try:
    ds = load_dataset("Salesforce/xlam-function-calling-60k", split="train")
    s = sample_n(ds, EVAL_SAMPLES)
    items = []
    for x in s:
        items.append({"q": f"Available tools:\n{x['tools']}\nUser query: {x['query']}\nWhich tool call (name only) should be made? Answer with just the function name.",
                       "a": x["answers"]})
    add_bench("Tool-Use", "xLAM Function-Calling", items)
except Exception as e:
    print(f"  ⚠️ xLAM failed: {e}")

print(f"\nВсего загружено категорий: {len(benchmarks)}")
for cat, dsets in benchmarks.items():
    print(f"  {cat}: {list(dsets.keys())}")


In [ ]:
# === 5. Универсальная функция оценки ответа ===
import re

def grade(pred, target):
    """Loose, deterministic grading heuristic (mirrors the style used in training notebook's eval cell)."""
    pred_l = str(pred).strip().lower()
    target_l = str(target).strip().lower()

    # valid/invalid
    if target_l in ["valid", "invalid"]:
        return int((target_l in pred_l) and not (target_l == "valid" and "invalid" in pred_l))

    # yes/no
    if target_l in ["yes", "no"]:
        return int(target_l in pred_l and not ("not " + target_l) in pred_l)

    # true/false/unknown/uncertain
    if target_l in ["true", "false", "uncertain", "unknown"]:
        t_map = "uncertain" if target_l in ["uncertain", "unknown"] else target_l
        p_clean = pred_l.replace("unknown", "uncertain")
        if t_map == "uncertain":
            return int("uncertain" in p_clean)
        elif t_map == "true":
            return int("true" in p_clean and "false" not in p_clean)
        else:
            return int("false" in p_clean and "true" not in p_clean)

    # single-letter multiple choice
    if len(target_l) == 1 and target_l.isalpha():
        patterns = [r'\b' + target_l + r'\b', r'\(' + target_l + r'\)', r'choice\s+' + target_l]
        for pat in patterns:
            if re.search(pat, pred_l):
                return 1
        return int(pred_l == target_l or target_l in pred_l.split())

    # numeric (GSM8K / MATH style)
    nums_target = re.findall(r"-?\d+\.?\d*", target_l)
    if nums_target:
        nums_pred = re.findall(r"-?\d+\.?\d*", pred_l)
        return int(bool(nums_pred) and nums_target[-1] in nums_pred)

    # fallback: substring containment (function names, free text)
    return int(target_l in pred_l)

print("✅ Grading function ready.")


In [ ]:
# === 6. Основной прогон: Base vs CIDA по всем бенчмаркам ===
from collections import defaultdict
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

raw_scores = defaultdict(lambda: defaultdict(list))
raw_latency = defaultdict(list)
sample_log = []

for variant_name, vcfg in VARIANTS_CONFIG.items():
    print(f"\n==========================================")
    print(f"🔄 EVALUATING VARIANT: {variant_name}")
    print(f"==========================================")
    
    free_memory()
    base_repo = vcfg["backbone"]
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    
    print(f"Loading tokenizer & backbone: {base_repo}...")
    tok = AutoTokenizer.from_pretrained(base_repo, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
        
    llm = AutoModelForCausalLM.from_pretrained(base_repo, quantization_config=bnb, device_map="auto")
    llm.eval()
    
    cida = None
    proj = None
    num_plan = 8
    
    if not vcfg["load_base_only"]:
        is_v4 = vcfg["is_v4"]
        is_hf = vcfg["is_hf"]
        source = vcfg["cida_source"]
        
        print(f"Loading CIDA Adapter from: {source}...")
        if is_v4:
            from cida_plugin.config import CIDAPluginConfig
            from cida_plugin.core import CIDAPlugin
            cfg = CIDAPluginConfig(
                d_input=llm.config.hidden_size, d_hidden=1024, d_message=1024,
                num_agents=4, num_classes=2, max_rounds=4,
                num_attn_heads=8, use_perspective_projector=True,
                anonymous_messages=True, sparse_comm_k=3, use_dynamic_trust=True
            )
            cida = CIDAPlugin(cfg).to(DEVICE).float().eval()
            cida.load_state_dict(torch.load(os.path.join(source, "pytorch_model.bin"), map_location="cpu"))
            
            proj = LatentAEProjector(cfg.d_hidden, llm.config.hidden_size, cfg.num_agents, num_plan=8).to(DEVICE).float().eval()
            proj.load_state_dict(torch.load(os.path.join(source, "projector.pt"), map_location="cpu"))
            num_plan = 8
        else:
            if is_hf:
                cfg_path = hf_hub_download(repo_id=source, filename="config.json")
                weights_path = hf_hub_download(repo_id=source, filename="pytorch_model.bin")
                proj_path = hf_hub_download(repo_id=source, filename="plan_projector.pt")
                cfg_dir = os.path.dirname(cfg_path)
            else:
                cfg_path = os.path.join(source, "config.json")
                weights_path = os.path.join(source, "pytorch_model.bin")
                proj_path = os.path.join(source, "plan_projector.pt")
                cfg_dir = source
            
            if "1.3b" in base_repo.lower() or "1.3b" in source.lower() or "v2" in source.lower():
                from cida_latent_plugin.config import CIDAPluginConfig as LatentConfig
                from cida_latent_plugin.core import CIDAPlugin as LatentCIDA
                cfg = LatentConfig.from_pretrained(cfg_dir)
                cida = LatentCIDA(cfg).to(DEVICE).float().eval()
            else:
                from cida_plugin.config import CIDAPluginConfig
                from cida_plugin.core import CIDAPlugin
                cfg = CIDAPluginConfig.from_pretrained(cfg_dir)
                cida = CIDAPlugin(cfg).to(DEVICE).float().eval()
                
            cida.load_state_dict(torch.load(weights_path, map_location="cpu"))
            
            proj_weights = torch.load(proj_path, map_location="cpu")
            out_features = proj_weights["projector.3.weight"].shape[0]
            num_plan = out_features // llm.config.hidden_size
            
            proj = CIDAPlanProjectorOld(cfg.d_hidden, llm.config.hidden_size, cfg.num_agents, num_plan_tokens=num_plan).to(DEVICE).float().eval()
            proj.load_state_dict(proj_weights)
            
        print(f"CIDA Adapter and Projector loaded successfully. Plan tokens: {num_plan}")
        
    is_cida_flag = not vcfg["load_base_only"]
    for category, dsets in benchmarks.items():
        for label, items in dsets.items():
            print(f"Running {category} / {label} ({len(items)} samples)...")
            for i, item in enumerate(items):
                text, dt = generate_single(
                    item["q"], llm, tok, cida=cida, proj=proj, 
                    is_cida=is_cida_flag, is_v4=vcfg.get("is_v4", False), 
                    num_plan=num_plan, max_new_tokens=150
                )
                score = grade(text, item["a"])
                raw_scores[category][variant_name].append(score)
                raw_latency[variant_name].append(dt)
                
                if i < 2:
                    sample_log.append({
                        "variant": variant_name,
                        "category": category,
                        "dataset": label,
                        "q": item["q"][:150],
                        "target": item["a"],
                        "output": text[:120],
                        "latency": dt
                    })
                    
            n = len(items)
            acc = 100.0 * sum(raw_scores[category][variant_name][-n:]) / max(n, 1)
            print(f"  -> {variant_name} accuracy: {acc:5.1f}%")
            
    del llm, tok
    if cida is not None:
        del cida, proj
    free_memory()

print("\n✅ Полный прогон завершён.")


In [ ]:
# === 7. Агрегация результатов и дашборд ===
import matplotlib.pyplot as plt
import numpy as np

variant_names = [v for v in VARIANTS_CONFIG.keys() if v in raw_latency]
categories = list(raw_scores.keys())

agg = {v: [] for v in variant_names}
for cat in categories:
    for v in variant_names:
        scores = raw_scores[cat][v]
        agg[v].append(100.0 * sum(scores) / max(len(scores), 1))

# --- bar chart per category ---
plt.style.use('dark_background')
x = np.arange(len(categories))
width = 0.8 / max(len(variant_names), 1)
colors = ["#475569", "#64748b", "#94a3b8", "#3b82f6", "#10b981", "#f59e0b", "#ec4899", "#8b5cf6"]

plt.figure(figsize=(max(10, 2*len(categories)), 6))
for i, v in enumerate(variant_names):
    plt.bar(x + i*width, agg[v], width, label=v, color=colors[i % len(colors)])
plt.xticks(x + width*(len(variant_names)-1)/2, categories, rotation=20, ha="right")
plt.ylabel("Accuracy (%)")
plt.title("CIDA Release-Readiness — Accuracy by Category")
plt.legend()
plt.ylim(0, 100)
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("cida_release_readiness_bars.png")
plt.show()

# --- radar chart ---
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, v in enumerate(variant_names):
    vals = agg[v] + agg[v][:1]
    ax.plot(angles, vals, label=v, color=colors[i % len(colors)], linewidth=2)
    ax.fill(angles, vals, color=colors[i % len(colors)], alpha=0.1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 100)
ax.set_title("Cognitive Profile: Base vs CIDA", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig("cida_release_readiness_radar.png")
plt.show()

print("📈 Saved: cida_release_readiness_bars.png, cida_release_readiness_radar.png")

# --- delta table ---
print("\n=== Δ vs respective Base LLM (percentage points) ===")
for cat_i, cat in enumerate(categories):
    line = f"{cat:35s}"
    for v in variant_names:
        vcfg = VARIANTS_CONFIG[v]
        if vcfg.get("load_base_only"):
            continue
        base_v = vcfg.get("base_variant")
        if base_v in variant_names:
            delta = agg[v][cat_i] - agg[base_v][cat_i]
            line += f" | {v} vs {base_v}: {delta:+5.1f}pp"
    print(line)


In [ ]:
# === 8. Латентность инференса (Base vs CIDA) ===
print("=== Average per-generation latency ===")
avg_latency = {v: (sum(raw_latency[v]) / max(len(raw_latency[v]), 1)) for v in variant_names}
for v in variant_names:
    print(f"  {v:>28s}: {avg_latency[v]*1000:7.1f} ms/generation  (n={len(raw_latency[v])})")

print("\n=== Overhead vs Respective Base LLM ===")
for v in variant_names:
    vcfg = VARIANTS_CONFIG[v]
    if vcfg.get("load_base_only"):
        continue
    base_v = vcfg.get("base_variant")
    if base_v in variant_names:
        overhead = (avg_latency[v] / avg_latency[base_v] - 1) * 100
        print(f"  ⏱️ Overhead {v} vs {base_v}: {overhead:+.1f}%")


In [ ]:
# === 9. Release-Readiness Scorecard (по твоим 6 критериям) ===
print("="*70)
print("   🏁 CIDA RELEASE READINESS SCORECARDS")
print("="*70)

THRESHOLDS = {
    "min_categories_with_gain": 5,
    "min_avg_gain_pp": 3.0,
    "max_latency_overhead_pct": 60.0,
    "max_categories_regressed": 2,
}

n_cats = len(categories)

for v in variant_names:
    vcfg = VARIANTS_CONFIG[v]
    if vcfg.get("load_base_only"):
        continue
    base_v = vcfg.get("base_variant")
    if base_v not in variant_names:
        continue
        
    print(f"\n--- SCORECARD FOR: {v} ---")
    gains = [agg[v][i] - agg[base_v][i] for i in range(n_cats)]
    n_gained = sum(1 for g in gains if g > 0)
    n_regressed = sum(1 for g in gains if g < -1.0)
    avg_gain = sum(gains) / max(n_cats, 1)
    overhead_pct = (avg_latency[v] / avg_latency[base_v] - 1) * 100
    
    checks = []
    checks.append(("Обобщение на несколько доменов (не только матем./код)",
                    n_gained >= THRESHOLDS["min_categories_with_gain"],
                    f"{n_gained}/{n_cats} категорий с приростом (нужно ≥{THRESHOLDS['min_categories_with_gain']})"))
    checks.append(("Средний прирост над Base значим",
                    avg_gain >= THRESHOLDS["min_avg_gain_pp"],
                    f"средний Δ = {avg_gain:+.1f}пп (нужно ≥{THRESHOLDS['min_avg_gain_pp']}пп)"))
    checks.append(("Нет серьёзных регрессий",
                    n_regressed <= THRESHOLDS["max_categories_regressed"],
                    f"{n_regressed} категорий с регрессом >1пп (допустимо ≤{THRESHOLDS['max_categories_regressed']})"))
    checks.append(("Инференс не слишком медленный",
                    overhead_pct <= THRESHOLDS["max_latency_overhead_pct"],
                    f"overhead = {overhead_pct:+.1f}% (нужно ≤{THRESHOLDS['max_latency_overhead_pct']}%))"))

    for name, passed, detail in checks:
        icon = "✅" if passed else "❌"
        print(f"{icon} {name}\n     -> {detail}")

    all_passed = all(p for _, p, _ in checks)
    print("-"*50)
    if all_passed:
        print(f"🟢 ВЕРДИКТ: {v} проходит формальный порог готовности к релизу.")
    else:
        print(f"🔴 ВЕРДИКТ: {v} ПОКА НЕ готов — см. непройденные пункты выше.")
    print("-"*50)

print("\nНапоминание по критериям, которые НЕ измеряются автоматически и требуют")
print("ручной/LLM-judge оценки:")
print("  • #1 Универсальность как адаптер (API-совместимость, архитектурная чистота)")
print("  • #4 Способность на нестандартные/творческие решения вне выученного домена")
print("  • #6 Лёгкость дообучения под новую задачу (нужен отдельный fine-tune-тест)")


## 🎨 Ручная/LLM-judge проверка нестандартности решений (критерий #4)

Автоматические бенчмарки выше проверяют **точность**, но не «придумал ли CIDA что-то нестандартное». Для этого ниже — небольшой набор open-ended задач без единственного правильного ответа (в духе «оптимизация CUDA», «новая эвристика»), где сравнивается **разнообразие/новизна** ответов всех доступных вариантов CIDA относительно их базовых моделей через семантическое расстояние (semantic distance по эмбеддингам Qwen2.5-1.5B-Instruct).

Это не даёт жёсткой метрики "готов/не готов" — это требует твоего суждения как автора продукта.


In [ ]:
# === 10. Открытые творческие задачи + метрика новизны через эмбеддинги ===
OPEN_ENDED_PROMPTS = [
    "Suggest a non-obvious heuristic to speed up a CUDA kernel that is memory-bandwidth bound.",
    "Propose an unconventional approach to reduce hallucination in a language model, other than RLHF or retrieval.",
    "Design a scoring rule for a multi-agent negotiation game that discourages collusion.",
    "Give a creative, non-standard way to compress a KV-cache without losing long-range dependencies.",
]

open_ended_outputs = defaultdict(dict)

for variant_name, vcfg in VARIANTS_CONFIG.items():
    if variant_name not in variant_names:
        continue
    print(f"Generating open-ended responses for: {variant_name}...")
    
    free_memory()
    base_repo = vcfg["backbone"]
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    tok = AutoTokenizer.from_pretrained(base_repo, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    llm = AutoModelForCausalLM.from_pretrained(base_repo, quantization_config=bnb, device_map="auto")
    llm.eval()
    
    cida = None
    proj = None
    num_plan = 8
    
    if not vcfg["load_base_only"]:
        is_v4 = vcfg["is_v4"]
        is_hf = vcfg["is_hf"]
        source = vcfg["cida_source"]
        if is_v4:
            from cida_plugin.config import CIDAPluginConfig
            from cida_plugin.core import CIDAPlugin
            cfg = CIDAPluginConfig(
                d_input=llm.config.hidden_size, d_hidden=1024, d_message=1024,
                num_agents=4, num_classes=2, max_rounds=4,
                num_attn_heads=8, use_perspective_projector=True,
                anonymous_messages=True, sparse_comm_k=3, use_dynamic_trust=True
            )
            cida = CIDAPlugin(cfg).to(DEVICE).float().eval()
            cida.load_state_dict(torch.load(os.path.join(source, "pytorch_model.bin"), map_location="cpu"))
            proj = LatentAEProjector(cfg.d_hidden, llm.config.hidden_size, cfg.num_agents, num_plan=8).to(DEVICE).float().eval()
            proj.load_state_dict(torch.load(os.path.join(source, "projector.pt"), map_location="cpu"))
        else:
            if is_hf:
                cfg_path = hf_hub_download(repo_id=source, filename="config.json")
                weights_path = hf_hub_download(repo_id=source, filename="pytorch_model.bin")
                proj_path = hf_hub_download(repo_id=source, filename="plan_projector.pt")
                cfg_dir = os.path.dirname(cfg_path)
            else:
                cfg_path = os.path.join(source, "config.json")
                weights_path = os.path.join(source, "pytorch_model.bin")
                proj_path = os.path.join(source, "plan_projector.pt")
                cfg_dir = source
                
            if "1.3b" in base_repo.lower() or "1.3b" in source.lower() or "v2" in source.lower():
                from cida_latent_plugin.config import CIDAPluginConfig as LatentConfig
                from cida_latent_plugin.core import CIDAPlugin as LatentCIDA
                cfg = LatentConfig.from_pretrained(cfg_dir)
                cida = LatentCIDA(cfg).to(DEVICE).float().eval()
            else:
                from cida_plugin.config import CIDAPluginConfig
                from cida_plugin.core import CIDAPlugin
                cfg = CIDAPluginConfig.from_pretrained(cfg_dir)
                cida = CIDAPlugin(cfg).to(DEVICE).float().eval()
                
            cida.load_state_dict(torch.load(weights_path, map_location="cpu"))
            proj_weights = torch.load(proj_path, map_location="cpu")
            out_features = proj_weights["projector.3.weight"].shape[0]
            num_plan = out_features // llm.config.hidden_size
            proj = CIDAPlanProjectorOld(cfg.d_hidden, llm.config.hidden_size, cfg.num_agents, num_plan_tokens=num_plan).to(DEVICE).float().eval()
            proj.load_state_dict(proj_weights)
            
    is_cida_flag = not vcfg["load_base_only"]
    for prompt in OPEN_ENDED_PROMPTS:
        text, dt = generate_single(
            prompt, llm, tok, cida=cida, proj=proj,
            is_cida=is_cida_flag, is_v4=vcfg.get("is_v4", False),
            num_plan=num_plan, max_new_tokens=200
        )
        open_ended_outputs[prompt][variant_name] = text
        
    del llm, tok
    if cida is not None:
        del cida, proj
    free_memory()

print("\nComputing semantic distances using Qwen2.5-1.5B-Instruct...")
free_memory()
eval_backbone = "Qwen/Qwen2.5-1.5B-Instruct"
tok_eval = AutoTokenizer.from_pretrained(eval_backbone)
llm_eval = AutoModelForCausalLM.from_pretrained(eval_backbone, torch_dtype=torch.float16, device_map="auto")
llm_eval.eval()

@torch.no_grad()
def embed_text(text):
    enc = tok_eval(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    out = llm_eval(input_ids=enc.input_ids, attention_mask=enc.attention_mask, output_hidden_states=True)
    return out.hidden_states[-1].mean(dim=1)[0].float()

def cos_sim(a, b):
    return torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print("=== Novelty probe: semantic distance CIDA-output vs Base-output ===")
for prompt in OPEN_ENDED_PROMPTS:
    print(f"\n📝 {prompt}")
    base_v_name = "Base Qwen2.5-1.5B"
    base_text = open_ended_outputs[prompt].get(base_v_name, "")
    if not base_text:
        base_v_name = [k for k in open_ended_outputs[prompt].keys() if "Base" in k][0]
        base_text = open_ended_outputs[prompt][base_v_name]
        
    base_emb = embed_text(base_text)
    print(f"  [{base_v_name}] {base_text[:160]}...")
    
    for v in variant_names:
        if v == base_v_name:
            continue
        cida_text = open_ended_outputs[prompt].get(v, "")
        if not cida_text:
            continue
        sim = cos_sim(base_emb, embed_text(cida_text))
        novelty = (1 - sim) * 100
        print(f"  [{v}] novelty vs {base_v_name}: {novelty:5.1f}%  -> {cida_text[:160]}...")

del llm_eval, tok_eval
free_memory()

print("\nℹ️ Высокая 'novelty' — не всегда хорошо (может значить бессвязность),")
print("   читай сами ответы, а не только цифру. Это качественный сигнал, не приговор.")


In [ ]:
# === 11. Экспорт полного отчёта в JSON (для истории релизов) ===
import json, datetime

report = {
    "timestamp": datetime.datetime.utcnow().isoformat(),
    "variants_evaluated": variant_names,
    "eval_samples_per_dataset": EVAL_SAMPLES,
    "accuracy_by_category": {cat: {v: agg[v][i] for v in variant_names} for i, cat in enumerate(categories)},
    "avg_latency_ms": {v: avg_latency[v]*1000 for v in variant_names},
    "sample_outputs": sample_log[:50],
}

with open("cida_release_readiness_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("✅ Отчёт сохранён: cida_release_readiness_report.json")
print("   (плюс PNG-дашборды из ячейки 7 — прикладывай их всё вместе как release-пакет)")
